# Multi-trainer Kaggle GPU notebook

This notebook runs the uploaded `multi_trainer-main` project on Kaggle using the notebook GPU runtime.

It is designed for the current project layout where:

- raw market data is copied/standardised with `src.kaggle_prepare_raw_data`
- features are generated with `src.prepare_mt5_data`
- side-specific direction datasets are generated with `src.prepare_direction_dataset`
- BUY-only and SELL-only models are trained through `local_run_pipeline_all_symbols.py`

Kaggle cannot run MetaTrader 5, so upload your existing raw M5 CSV data as a Kaggle Dataset. The notebook will prepare those CSVs into the project format and write trained models/logs to `/kaggle/working/run_outputs`.


## Kaggle setup checklist

1. Create or open a Kaggle notebook.
2. In **Settings**, enable **Accelerator → GPU**.
3. Add a Kaggle Dataset containing this project zip, for example `multi_trainer-main.zip`.
4. Add a Kaggle Dataset containing your raw M5 CSV files. Either use one CSV per symbol or one combined CSV with a `symbol`/`ticker`/`pair` column.
5. Run the cells from top to bottom.

The notebook defaults to a small smoke test so you can check that the data paths and GPU work. After the smoke test succeeds, set `RUN_PROFILE = "full_train"` in the settings cell.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import textwrap
import zipfile

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

print('Python:', sys.version)
print('Kaggle input exists:', KAGGLE_INPUT.exists())
print('Kaggle working:', KAGGLE_WORKING)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle input exists: True
Kaggle working: /kaggle/working


## 1. Configure the run

Leave `RUN_PROFILE = "smoke_test"` for the first run. It trains one symbol, one architecture and both sides for a very small number of epochs.

Change to `RUN_PROFILE = "full_train"` once the notebook has proven that it can see your raw data and the GPU.


In [32]:
# -------------------------
# Main user settings
# -------------------------
smoke_test = "smoke_test"
single_symbol = "single_symbol"
full_train = "full_train"
RUN_PROFILE = full_train  # "smoke_test", "single_symbol", or "full_train"
TIMEFRAME = "M5"

# The notebook auto-finds this zip under /kaggle/input.
# Override CODE_ZIP_PATH if your filename is different.
CODE_ZIP_NAME = "multi_trainer-main.zip"
CODE_ZIP_PATH = "/kaggle/input/datasets/chrisrose100/mt5-dataset/multi_trainer-main.zip"  # e.g. "/kaggle/input/my-code-dataset/multi_trainer-main.zip"

# Raw data input.
# Leave RAW_INPUT_DIR=None to auto-detect a /kaggle/input dataset folder containing CSVs.
RAW_INPUT_DIR = "/kaggle/input/datasets/chrisrose100/mt5-dataset/raw"  # e.g. "/kaggle/input/my-forex-m5-csvs"

# Optional: use this if your raw CSVs are inside a zip dataset file.
RAW_DATA_ZIP_PATH = None  # e.g. "/kaggle/input/my-forex-m5-csvs/raw_csvs.zip"

# Optional: set this if you have one combined CSV containing a symbol/ticker/pair column.
COMBINED_CSV_PATH = None  # e.g. "/kaggle/input/my-forex-m5-csvs/all_symbols_M5.csv"

# Your reduced 10-symbol Forex universe from this code branch.
DEFAULT_SYMBOLS = [
    "USDCAD", "EURGBP", "EURUSD", "USDJPY", "USDCHF",
    "GBPUSD", "AUDCHF", "GBPCHF", "AUDNZD", "EURCHF",
]

ALL_MODEL_CONFIGS = [
    "config/direction_settings_residual_mlp.yaml",
    "config/direction_settings_tcn.yaml",
    "config/direction_settings_inception_time.yaml",
    "config/direction_settings_small_transformer.yaml",
    "config/direction_settings_mixture_of_experts.yaml",
    "config/direction_settings_llm_transformer.yaml",
    "config/direction_settings_timesnet.yaml",
    "config/direction_settings_tsmixer.yaml",
]

# Train/replay date split. Adjust to match the dates in your uploaded CSVs.
TRAIN_START = "2022-01-01"
TRAIN_END = "2025-01-01"
REPLAY_START = "2025-01-01"
REPLAY_END = "2026-01-01"

# GPU/session safeguards.
# Kaggle normally gives a single GPU, so keep PARALLEL_JOBS=1 unless you know the model fits with more.
PARALLEL_JOBS = 1
PREPARE_WORKERS = 2
FORCE_REBUILD_DATA = True
DRY_RUN = False

# Optional row limits. Use these for smoke tests or when testing paths.
RAW_MAX_ROWS_PER_SYMBOL = None
FEATURE_MAX_ROWS = None
DIRECTION_MAX_ROWS = None
TRAIN_MAX_ROWS = None

if RUN_PROFILE == "smoke_test":
    TRAIN_SYMBOLS = ["EURUSD"]
    MODEL_CONFIGS = ["config/direction_settings_residual_mlp.yaml"]
    SIDES = ["buy", "sell"]
    EPOCHS = 2
    BATCH_SIZE = 256
    RAW_MAX_ROWS_PER_SYMBOL = 120_000
    FEATURE_MAX_ROWS = 120_000
    DIRECTION_MAX_ROWS = 120_000
    TRAIN_MAX_ROWS = 120_000
elif RUN_PROFILE == "single_symbol":
    TRAIN_SYMBOLS = ["EURUSD"]
    MODEL_CONFIGS = ALL_MODEL_CONFIGS
    SIDES = ["buy", "sell"]
    EPOCHS = 50
    BATCH_SIZE = 512
elif RUN_PROFILE == "full_train":
    TRAIN_SYMBOLS = DEFAULT_SYMBOLS
    MODEL_CONFIGS = ALL_MODEL_CONFIGS
    SIDES = ["buy", "sell"]
    EPOCHS = 50
    BATCH_SIZE = 512
else:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

print('Run profile:', RUN_PROFILE)
print('Symbols:', TRAIN_SYMBOLS)
print('Model configs:', MODEL_CONFIGS)
print('Sides:', SIDES)
print('Epochs:', EPOCHS, 'Batch size:', BATCH_SIZE)


Run profile: full_train
Symbols: ['USDCAD', 'EURGBP', 'EURUSD', 'USDJPY', 'USDCHF', 'GBPUSD', 'AUDCHF', 'GBPCHF', 'AUDNZD', 'EURCHF']
Model configs: ['config/direction_settings_residual_mlp.yaml', 'config/direction_settings_tcn.yaml', 'config/direction_settings_inception_time.yaml', 'config/direction_settings_small_transformer.yaml', 'config/direction_settings_mixture_of_experts.yaml', 'config/direction_settings_llm_transformer.yaml', 'config/direction_settings_timesnet.yaml', 'config/direction_settings_tsmixer.yaml']
Sides: ['buy', 'sell']
Epochs: 50 Batch size: 512


## 2. Check GPU and install non-Torch dependencies

Kaggle normally already has a CUDA-enabled PyTorch build. This cell deliberately avoids reinstalling `torch`, because replacing Kaggle's preinstalled CUDA build can break GPU detection.


In [22]:
import importlib.util

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        DEVICE = 'cuda'
    else:
        print('WARNING: No GPU detected. Check Kaggle notebook Settings -> Accelerator -> GPU.')
        DEVICE = 'cpu'
except Exception as exc:
    print('Torch import failed:', repr(exc))
    DEVICE = 'cpu'

def package_missing(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is None

missing = []
for import_name, pip_name in [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('yaml', 'PyYAML'),
    ('sklearn', 'scikit-learn'),
    ('joblib', 'joblib'),
]:
    if package_missing(import_name):
        missing.append(pip_name)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All non-Torch requirements are already available.')

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
All non-Torch requirements are already available.


'1'

## 3. Extract the project code

The notebook looks for `multi_trainer-main.zip` under `/kaggle/input`. Override `CODE_ZIP_PATH` in the settings cell if needed.


In [33]:
def find_file_under_input(filename: str) -> Path:
    hits = sorted(KAGGLE_INPUT.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename!r} under /kaggle/input. "
            "Add the code zip as a Kaggle Dataset or set CODE_ZIP_PATH explicitly."
        )
    if len(hits) > 1:
        print('Multiple matching code zips found; using:', hits[0])
        for h in hits:
            print('  ', h)
    return hits[0]

code_zip = Path(CODE_ZIP_PATH) if CODE_ZIP_PATH else find_file_under_input(CODE_ZIP_NAME)
print('Using code zip:', code_zip)

PROJECT_PARENT = KAGGLE_WORKING / 'project_src'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

# Clean old extraction so reruns are predictable.
for old in PROJECT_PARENT.glob('*'):
    if old.is_dir():
        shutil.rmtree(old)
    else:
        old.unlink()

with zipfile.ZipFile(code_zip, 'r') as zf:
    zf.extractall(PROJECT_PARENT)

project_candidates = [p for p in PROJECT_PARENT.iterdir() if p.is_dir() and (p / 'src').exists()]
if not project_candidates:
    raise RuntimeError(f'Could not find extracted project root with src/ under {PROJECT_PARENT}')
PROJECT_ROOT = project_candidates[0]
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('Project files:', [p.name for p in PROJECT_ROOT.iterdir() if p.name in ['src', 'config', 'local_run_pipeline_all_symbols.py', 'requirements.txt']])


Using code zip: /kaggle/input/datasets/chrisrose100/mt5-dataset/multi_trainer-main/multi_trainer-main.zip


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/chrisrose100/mt5-dataset/multi_trainer-main/multi_trainer-main.zip'

## 4. Find or extract the raw CSV data

Supported inputs:

- one CSV per symbol, for example `EURUSD_M5.csv`, `GBPUSD_M5.csv`
- one combined CSV with a symbol-like column such as `symbol`, `ticker`, `pair`
- optionally a zip containing CSVs, if `RAW_DATA_ZIP_PATH` is set


In [ ]:
def contains_csvs(path: Path) -> bool:
    return path.exists() and any(p.suffix.lower() == '.csv' for p in path.rglob('*.csv'))

if RAW_DATA_ZIP_PATH:
    raw_zip = Path(RAW_DATA_ZIP_PATH)
    if not raw_zip.exists():
        raise FileNotFoundError(raw_zip)
    raw_extract_dir = KAGGLE_WORKING / 'raw'
    if raw_extract_dir.exists():
        shutil.rmtree(raw_extract_dir)
    raw_extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(raw_zip, 'r') as zf:
        zf.extractall(raw_extract_dir)
    RAW_DIR = raw_extract_dir
elif RAW_INPUT_DIR:
    RAW_DIR = Path(RAW_INPUT_DIR)
else:
    # Prefer folders containing CSVs. This works when the raw CSV dataset is separate
    # or when it is the same dataset as the code zip.
    candidates = []
    for d in sorted(KAGGLE_INPUT.iterdir()):
        if d.is_dir() and contains_csvs(d):
            candidates.append(d)
    if not candidates:
        raise FileNotFoundError(
            'Could not auto-detect a raw CSV folder under /kaggle/input. '
            'Set RAW_INPUT_DIR or RAW_DATA_ZIP_PATH in the settings cell.'
        )
    RAW_DIR = candidates[0]
    if len(candidates) > 1:
        print('Multiple raw CSV folders found; using:', RAW_DIR)
        for c in candidates:
            print('  ', c)

print('Raw data folder:', RAW_DIR)
print('CSV examples:')
for p in sorted(RAW_DIR.rglob('*.csv'))[:20]:
    print('  ', p)

if COMBINED_CSV_PATH:
    print('Combined CSV:', COMBINED_CSV_PATH)


## 5. Run the Kaggle training pipeline

This uses `local_run_pipeline_all_symbols.py` so that raw-data preparation, feature preparation, direction-label generation, per-side config creation and train+replay are all run consistently.

Outputs are written under `/kaggle/working/run_outputs`.


In [ ]:
OUTPUT_ROOT = KAGGLE_WORKING / 'run_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, 'local_run_pipeline_all_symbols.py',
    '--raw-input-dir', str(RAW_DIR),
    '--symbols', *TRAIN_SYMBOLS,
    '--timeframe', TIMEFRAME,
    '--configs', *MODEL_CONFIGS,
    '--sides', *SIDES,
    '--mode', 'train-side-all',
    '--train-start', TRAIN_START,
    '--train-end', TRAIN_END,
    '--replay-start', REPLAY_START,
    '--replay-end', REPLAY_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
    '--parallel-jobs', str(PARALLEL_JOBS),
    '--prepare-workers', str(PREPARE_WORKERS),
    '--output-root', str(OUTPUT_ROOT),
]

if COMBINED_CSV_PATH:
    cmd.extend(['--combined-csv', str(COMBINED_CSV_PATH)])
if RAW_MAX_ROWS_PER_SYMBOL:
    cmd.extend(['--raw-max-rows-per-symbol', str(RAW_MAX_ROWS_PER_SYMBOL)])
if FEATURE_MAX_ROWS:
    cmd.extend(['--feature-max-rows', str(FEATURE_MAX_ROWS)])
if DIRECTION_MAX_ROWS:
    cmd.extend(['--direction-max-rows', str(DIRECTION_MAX_ROWS)])
if TRAIN_MAX_ROWS:
    cmd.extend(['--train-max-rows', str(TRAIN_MAX_ROWS)])
if FORCE_REBUILD_DATA:
    cmd.extend(['--force-raw', '--force-features'])
if DRY_RUN:
    cmd.append('--dry-run')

print('Command:')
print(' '.join(map(str, cmd)))

# Stream output live so Kaggle does not look idle during long training.
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
rc = process.wait()
print('\nReturn code:', rc)
if rc != 0:
    raise RuntimeError(f'Pipeline failed with return code {rc}')


## 6. Inspect the training summary

The pipeline writes a JSON summary containing every model/symbol/side task and stdout/stderr tails for any failures.


In [ ]:
summary_path = OUTPUT_ROOT / 'logs' / 'local_side_setup_training_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Summary:', summary_path)
    print('Tasks:', len(summary.get('tasks', [])))
    print('Failures:', len(summary.get('failures', [])))
    if summary.get('failures'):
        print('\nFailure preview:')
        for failure in summary['failures'][:5]:
            print(json.dumps({
                'symbol': failure.get('symbol'),
                'side': failure.get('side'),
                'config': failure.get('config'),
                'returncode': failure.get('returncode'),
                'stderr_tail': failure.get('stderr_tail', '')[-1000:],
            }, indent=2))
else:
    print('No summary found yet:', summary_path)

print('\nModel output folders:')
for p in sorted((OUTPUT_ROOT / 'models').glob('*'))[:20]:
    print(' ', p)


## 7. Package outputs for download

This creates `/kaggle/working/multi_trainer_kaggle_outputs.zip`, containing models, logs, generated configs and prepared data that you may want to reuse.


In [ ]:
package_root = KAGGLE_WORKING / 'package_outputs'
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)

for name in ['models', 'logs', 'config']:
    src = OUTPUT_ROOT / name
    if src.exists():
        shutil.copytree(src, package_root / name)

# Include prepared data if present. This is useful if you want to resume/replay without regenerating.
for rel in ['data/raw', 'data/processed_m5', 'data/direction']:
    src = PROJECT_ROOT / rel
    if src.exists():
        dst = package_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)

zip_base = KAGGLE_WORKING / 'multi_trainer_kaggle_outputs'
zip_path = shutil.make_archive(str(zip_base), 'zip', package_root)
print('Created:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / 1024 / 1024)


## Optional: prepare data only

Use this command instead of the training cell if you only want Kaggle to standardise raw CSVs, prepare features and generate direction-label CSVs. This is useful before a long full training run.


In [ ]:
# Uncomment and run manually if you want prepare-only mode.
# prepare_cmd = [
#     sys.executable, 'local_run_pipeline_all_symbols.py',
#     '--raw-input-dir', str(RAW_DIR),
#     '--symbols', *TRAIN_SYMBOLS,
#     '--timeframe', TIMEFRAME,
#     '--configs', *MODEL_CONFIGS,
#     '--mode', 'prepare-only',
#     '--train-start', TRAIN_START,
#     '--replay-end', REPLAY_END,
#     '--prepare-workers', str(PREPARE_WORKERS),
#     '--output-root', str(OUTPUT_ROOT),
# ]
# if COMBINED_CSV_PATH:
#     prepare_cmd.extend(['--combined-csv', str(COMBINED_CSV_PATH)])
# print(' '.join(map(str, prepare_cmd)))
# subprocess.check_call(prepare_cmd)
